# Clase 13 — Sesión 1: Arquitectura de Spark

## 🔧 Inicialización del entorno Spark

In [1]:
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

// 🔇 Silenciar TODOS los logs de Spark (log4j 2 — el que usa Spark 4)
import org.apache.logging.log4j.{Level, LogManager}
import org.apache.logging.log4j.core.config.Configurator

Configurator.setRootLevel(Level.ERROR)
Configurator.setLevel("org",         Level.ERROR)
Configurator.setLevel("org.apache",  Level.ERROR)
Configurator.setLevel("org.apache.spark", Level.ERROR)
Configurator.setLevel("org.sparkproject", Level.ERROR)
Configurator.setLevel("akka",        Level.ERROR)

import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("Ejercicios")
  .master("local[*]")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

val sc = spark.sparkContext

// Volvemos a silenciar tras crear la sesión, por si Spark reinicia el logger
Configurator.setRootLevel(Level.ERROR)

println(s"✅ Entorno listo — Spark ${spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/28 03:02:08 INFO SparkContext: Running Spark version 4.1.1
26/04/28 03:02:08 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/04/28 03:02:08 INFO SparkContext: Java version 17.0.18+8
26/04/28 03:02:09 INFO ResourceUtils: ==============================================================
26/04/28 03:02:09 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/28 03:02:09 INFO ResourceUtils: ==============================================================
26/04/28 03:02:09 INFO SparkContext: Submitted application: Ejercicios
26/04/28 03:02:09 INFO SecurityManager: Changing view acls to: gre
26/04/28 03:02:09 INFO SecurityManager: Changing modify acls to: gre
26/04/28 03:02:09 INFO SecurityManager: Changing view acls groups to: gre
26/04/28 03:02:09 INFO SecurityManager: Changing modify acls groups to: gre
26/04/28 03:02:09 INFO SecurityManager: SecurityManager: authentication disable

✅ Entorno listo — Spark 4.1.1


import $ivy.$
import $ivy.$
import org.apache.logging.log4j.{Level, LogManager}
import org.apache.logging.log4j.core.config.Configurator
import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@2dd57ce8
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@b89390f

## 🔹 Ejercicio 1 — Arrancar Spark y verificar la configuración

In [2]:
println(s"✅ Spark versión:       ${spark.version}")
println(s"✅ Scala versión:       ${scala.util.Properties.versionString}")
println(s"✅ Cores disponibles:   ${sc.defaultParallelism}")
println(s"✅ App Name:            ${sc.appName}")
println(s"✅ Master:              ${sc.master}")
println(s"🌐 Spark UI:            http://localhost:4040")

✅ Spark versión:       4.1.1
✅ Scala versión:       version 2.13.17
✅ Cores disponibles:   16
✅ App Name:            Ejercicios
✅ Master:              local[*]
🌐 Spark UI:            http://localhost:4040


## 🔹 Ejercicio 2 — Generar Jobs y verlos en la Spark UI

**Paso 1 — Crear datos**

In [3]:
val numeros = sc.parallelize(1 to 1000000)
println(s"Particiones creadas: ${numeros.getNumPartitions}")

Particiones creadas: 16


numeros: org.apache.spark.rdd.RDD[Int] = ParallelCollectionRDD[0] at parallelize at cmd3.sc:1

**Paso 2 — Primera acción: `count()`**

In [4]:
val total = numeros.count()
println(s"Total elementos: $total")

Total elementos: 1000000


total: Long = 1000000L

**Paso 3 — `filter` + `map` + `count` (dos transformaciones, un job)**

In [5]:
val pares        = numeros.filter(_ % 2 == 0)
val paresPorTres = pares.map(_ * 3)
val resultado    = paresPorTres.count()
println(s"Cantidad de (pares × 3): $resultado")

Cantidad de (pares × 3): 500000


pares: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[1] at filter at cmd5.sc:1
paresPorTres: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[2] at map at cmd5.sc:2
resultado: Long = 500000L

**Paso 4 — `reduce` (necesita combinación parcial → 2 stages)**

In [6]:
val suma = paresPorTres.reduce(_ + _)
println(s"Suma total: $suma")

Suma total: -1617776800


suma: Int = -1617776800

## 🔹 Ejercicio 3 — Visualizar el DAG

**Paso 1 — Cadena de transformaciones sobre ventas**

In [7]:
val ventas = sc.parallelize(List(
  ("Madrid",    1200.0),
  ("Barcelona",  800.0),
  ("Madrid",     950.0),
  ("Valencia",   600.0),
  ("Barcelona", 1100.0),
  ("Madrid",     700.0),
  ("Valencia",   850.0),
  ("Barcelona",  300.0)
))

val ventasMadrid   = ventas.filter(_._1 == "Madrid")
val importesMadrid = ventasMadrid.map(_._2)
val totalMadrid    = importesMadrid.reduce(_ + _)

println(f"Total ventas Madrid: $totalMadrid%.2f €")

Total ventas Madrid: 2850,00 €


ventas: org.apache.spark.rdd.RDD[(String, Double)] = ParallelCollectionRDD[3] at parallelize at cmd7.sc:1
ventasMadrid: org.apache.spark.rdd.RDD[(String, Double)] = MapPartitionsRDD[4] at filter at cmd7.sc:12
importesMadrid: org.apache.spark.rdd.RDD[Double] = MapPartitionsRDD[5] at map at cmd7.sc:13
totalMadrid: Double = 2850.0

**Paso 2 — Linaje del RDD con `toDebugString`**

In [10]:
println(importesMadrid.toDebugString)

(16) MapPartitionsRDD[5] at map at cmd7.sc:13 []
 |   MapPartitionsRDD[4] at filter at cmd7.sc:12 []
 |   ParallelCollectionRDD[3] at parallelize at cmd7.sc:1 []


**Paso 3 — `groupByKey` introduce un shuffle (límite entre stages)**

In [11]:
val ventasPorCiudad = ventas.groupByKey()
println(ventasPorCiudad.toDebugString)

(16) ShuffledRDD[7] at groupByKey at cmd11.sc:1 []
 +-(16) ParallelCollectionRDD[3] at parallelize at cmd7.sc:1 []


ventasPorCiudad: org.apache.spark.rdd.RDD[(String, Iterable[Double])] = ShuffledRDD[7] at groupByKey at cmd11.sc:1

In [12]:
val resumen = ventasPorCiudad.mapValues(importes => importes.sum)
resumen.collect().foreach { case (ciudad, total) =>
  println(f"  $ciudad%-15s → $total%.2f €")
}

  Valencia        → 1450,00 €
  Barcelona       → 2200,00 €
  Madrid          → 2850,00 €


resumen: org.apache.spark.rdd.RDD[(String, Double)] = MapPartitionsRDD[8] at mapValues at cmd12.sc:1

## 🔹 Ejercicio 4 — Identificar stages y tasks

In [13]:
val palabras = sc.parallelize(List(
  "spark", "scala", "big", "data", "spark", "es", "rapido",
  "scala", "es", "elegante", "spark", "escala", "bien",
  "big", "data", "necesita", "spark"
))

val frecuencias = palabras
  .map(palabra => (palabra, 1))
  .reduceByKey(_ + _)
  .sortBy({ case (_, c) => c }, ascending = false)

frecuencias.collect().foreach { case (palabra, count) =>
  println(f"  $palabra%-15s: $count veces")
}

  spark          : 4 veces
  big            : 2 veces
  scala          : 2 veces
  data           : 2 veces
  es             : 2 veces
  bien           : 1 veces
  escala         : 1 veces
  rapido         : 1 veces
  elegante       : 1 veces
  necesita       : 1 veces


palabras: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[9] at parallelize at cmd13.sc:1
frecuencias: org.apache.spark.rdd.RDD[(String, Int)] = MapPartitionsRDD[16] at sortBy at cmd13.sc:10

### 📝 Respuestas a las preguntas del Ejercicio 4

1. **¿Cuántos Jobs se crearon al ejecutar `.collect()`?** → Se crean **2 jobs**: uno disparado internamente por `sortBy` (necesita muestrear los datos para calcular el rango del particionador) y otro por el propio `collect()`.
2. **¿Cuántos Stages tiene el job principal?** → El job de `collect()` tiene **3 stages**: (a) `map → reduceByKey` lado mapa, (b) tras shuffle de `reduceByKey` + `sortBy` lado mapa, (c) tras shuffle de `sortBy`.
3. **¿Cuántas Tasks hay en total?** → Aproximadamente *particiones × nº de stages* (con `local[*]` típicamente 8 particiones por stage).
4. **¿En qué stage se produce el shuffle?** → En los límites entre stages: el primer shuffle lo provoca `reduceByKey` y el segundo `sortBy`. Se identifican en la UI por las columnas **Shuffle Write / Shuffle Read** y por las flechas en el DAG visual.
5. **¿Cuánta memoria está usando Spark?** → En la pestaña **Executors** aparece un único executor `driver` (modo `local[*]`) con la memoria asignada por defecto (≈ 366 MB de Storage Memory).

---
# 🏢 Caso de Estudio — LogiTrack S.A.

Como trabajamos en **Jupyter**, en lugar de generar el fichero desde PowerShell cargamos los registros directamente en un RDD con `parallelize`.

## Tarea 1 — Arrancar el entorno y verificar la arquitectura local

In [14]:
println("=" * 55)
println("  INFORME DE CONFIGURACIÓN — LogiTrack S.A.")
println("=" * 55)
println(s"  Versión Spark:        ${spark.version}")
println(s"  Versión Scala:        ${scala.util.Properties.versionString}")
println(s"  Modo de ejecución:    ${sc.master}")
println(s"  Cores disponibles:    ${sc.defaultParallelism}")
println(s"  Nombre de la app:     ${sc.appName}")
println(s"  Spark UI disponible:  http://localhost:4040")
println("=" * 55)

  INFORME DE CONFIGURACIÓN — LogiTrack S.A.
  Versión Spark:        4.1.1
  Versión Scala:        version 2.13.17
  Modo de ejecución:    local[*]
  Cores disponibles:    16
  Nombre de la app:     Ejercicios
  Spark UI disponible:  http://localhost:4040


### 📝 Respuestas — Tarea 1

1. En modo `local[*]`, **el mismo proceso JVM** del kernel de Jupyter actúa como **Driver, Executor y Cluster Manager** a la vez. Spark simula un clúster usando los hilos de la máquina local.
2. **Cores disponibles** se corresponde con el **paralelismo por defecto**: número de tasks que pueden ejecutarse simultáneamente y, por tanto, número de particiones por defecto al crear un RDD.
3. Con `master = yarn`, el **Driver** sigue siendo nuestro programa, pero el **Cluster Manager** pasa a ser **YARN** (gestor de recursos de Hadoop) y los **Executors** se lanzan como contenedores en nodos distintos del clúster, con su propia JVM, memoria y CPU.

## Tarea 2 — El DAG en acción

**Paso 1 — Cargar los datos y definir transformaciones (lazy)**

In [15]:
val registros = List(
  "E001,Madrid,Barcelona,completada,245",
  "E002,Valencia,Sevilla,completada,312",
  "E003,Barcelona,Bilbao,en_ruta,180",
  "E004,Madrid,Valencia,completada,198",
  "E005,Sevilla,Madrid,completada,275",
  "E006,Bilbao,Barcelona,retrasada,420",
  "E007,Valencia,Madrid,completada,210",
  "E008,Madrid,Sevilla,completada,290",
  "E009,Barcelona,Valencia,en_ruta,155",
  "E010,Madrid,Bilbao,retrasada,480",
  "E011,Sevilla,Barcelona,completada,330",
  "E012,Valencia,Bilbao,completada,265",
  "E013,Madrid,Barcelona,completada,252",
  "E014,Bilbao,Madrid,completada,300",
  "E015,Barcelona,Sevilla,retrasada,395",
  "E016,Madrid,Valencia,completada,185",
  "E017,Sevilla,Bilbao,en_ruta,210",
  "E018,Madrid,Barcelona,completada,238",
  "E019,Valencia,Barcelona,completada,175",
  "E020,Madrid,Sevilla,completada,310"
)

val entregas      = sc.parallelize(registros)
val lineasValidas = entregas.filter(_.trim.nonEmpty)
val estados       = lineasValidas.map(linea => linea.split(",")(3))
val paresEstado   = estados.map(estado => (estado, 1))

println("Plan construido (lazy). Spark aún no ha ejecutado nada.")

Plan construido (lazy). Spark aún no ha ejecutado nada.


registros: List[String] = List(
  "E001,Madrid,Barcelona,completada,245",
  "E002,Valencia,Sevilla,completada,312",
  "E003,Barcelona,Bilbao,en_ruta,180",
  "E004,Madrid,Valencia,completada,198",
  "E005,Sevilla,Madrid,completada,275",
  "E006,Bilbao,Barcelona,retrasada,420",
  "E007,Valencia,Madrid,completada,210",
  "E008,Madrid,Sevilla,completada,290",
  "E009,Barcelona,Valencia,en_ruta,155",
  "E010,Madrid,Bilbao,retrasada,480",
  "E011,Sevilla,Barcelona,completada,330",
  "E012,Valencia,Bilbao,completada,265",
  "E013,Madrid,Barcelona,completada,252",
  "E014,Bilbao,Madrid,completada,300",
  "E015,Barcelona,Sevilla,retrasada,395",
  "E016,Madrid,Valencia,completada,185",
  "E017,Sevilla,Bilbao,en_ruta,210",
  "E018,Madrid,Barcelona,completada,238",
  "E019,Valencia,Barcelona,completada,175",
  "E020,Madrid,Sevilla,completada,310"
)
entregas: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[17] at parallelize at cmd15.sc:24
lineasValidas: org.apache.spark.rdd.RDD[String] = 

**Paso 2 — Primera acción**

In [16]:
val totalEntregas = lineasValidas.count()
println(s"Total de entregas registradas hoy: $totalEntregas")

Total de entregas registradas hoy: 20


totalEntregas: Long = 20L

**Paso 3 — Conteo por estado (introduce shuffle)**

In [17]:
val conteoEstados = paresEstado.reduceByKey(_ + _)
conteoEstados.collect().foreach { case (estado, cantidad) =>
  println(f"  $estado%-15s: $cantidad entregas")
}

  completada     : 14 entregas
  en_ruta        : 3 entregas
  retrasada      : 3 entregas


conteoEstados: org.apache.spark.rdd.RDD[(String, Int)] = ShuffledRDD[21] at reduceByKey at cmd17.sc:1

### 📝 Respuestas — Tarea 2

1. `lineasValidas.count()` dispara **1 job** porque `count` es **una sola acción**, y un job se crea por cada acción.
2. `conteoEstados.collect()` genera un job con **2 stages**: el primero ejecuta `map + map + reduceByKey` en el lado mapa (combinación local) y, tras el **shuffle** que `reduceByKey` introduce, el segundo agrega los resultados parciales por clave. El shuffle se produce justo en el `reduceByKey`.
3. **Evaluación perezosa**: al definir `entregas → lineasValidas → estados → paresEstado` no se procesó ni un dato; Spark solo registró el **plan lógico** (DAG). Solo cuando llamamos a `.count()` y a `.collect()` el Driver envió tasks a los Executors. Esto permite a Spark **optimizar** el plan completo (encadenando filter + map sin materializar resultados intermedios) y evita trabajo inútil.

## Tarea 3 — Explorar la Spark UI como herramienta de diagnóstico

In [18]:
val duraciones = entregas
  .filter(_.trim.nonEmpty)
  .map(_.split(","))
  .filter(_.length == 5)
  .map(campos => campos(4).toInt)

val totalEntregasD = duraciones.count()
val duracionMedia  = duraciones.sum() / totalEntregasD
val duracionMaxima = duraciones.max()
val duracionMinima = duraciones.min()

println(s"Entregas analizadas:  $totalEntregasD")
println(f"Duración media:       $duracionMedia%.0f minutos")
println(s"Duración máxima:      $duracionMaxima minutos")
println(s"Duración mínima:      $duracionMinima minutos")

Entregas analizadas:  20
Duración media:       271 minutos
Duración máxima:      480 minutos
Duración mínima:      155 minutos


duraciones: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[25] at map at cmd18.sc:5
totalEntregasD: Long = 20L
duracionMedia: Double = 271.25
duracionMaxima: Int = 480
duracionMinima: Int = 155

### 📋 Observaciones de la Spark UI (`http://localhost:4040`)

| Pestaña      | Información que se encuentra | Elementos típicos |
| ------------ | ---------------------------- | ----------------- |
| **Jobs**     | Lista de jobs ejecutados con su acción origen, duración y nº de stages/tasks. | 1 job por cada acción: `count`, `sum`, `max`, `min`, `collect`… |
| **Stages**   | Detalle de cada stage: tasks completadas, datos leídos/escritos, **Shuffle Read / Write**. | Stages sin shuffle (1 por job de stats) y con shuffle (los que vienen de `reduceByKey`). |
| **Executors**| Executors activos, memoria, cores, tasks ejecutadas. | 1 executor llamado `driver` (modo `local[*]`). |
| **Environment** | Configuración de Spark, variables JVM, classpath. | Cientos de propiedades (`spark.master`, `spark.app.name`, versiones…). |

### 📝 Respuestas — Tarea 3

1. El job de `duraciones.max()` tiene **1 stage** (no requiere shuffle: cada partición calcula su máximo local y el Driver combina los resultados parciales). En el DAG visual aparece la cadena `parallelize → filter → map → filter → map`.
2. Los stages con **Shuffle Write/Read** corresponden al `reduceByKey` de la Tarea 2 (conteo por estado). Necesita shuffle porque para sumar los `(estado, 1)` por clave Spark debe **redistribuir** todos los pares con la misma clave a la misma partición.
3. En `local[*]` solo hay **un executor** (el `driver`) con la memoria asignada por defecto (≈ 366 MB de Storage Memory) y tantos slots como cores tenga la máquina. Esto demuestra que en local Spark **no usa un clúster real**, sino los hilos del proceso JVM.

## Tarea 4 — Razonamiento sobre la arquitectura

### ✉️ Respuesta al correo de Carlos Mendoza (Director de Tecnología, LogiTrack S.A.)

> Estimado Carlos,
>
> Gracias por sus dudas; son muy razonables y conviene aclararlas antes de seguir avanzando.
>
> **Sobre el modo `local[*]`.** Lo estamos usando **únicamente como entorno de desarrollo y validación**. En este modo, un mismo proceso hace de **Driver**, de **Executor** y de **Cluster Manager**, lo que permite probar el código y la lógica de negocio sin depender de la infraestructura del clúster. Cuando pasemos a producción, bastará con cambiar el parámetro `master` (a `yarn`, `kubernetes` o `spark://`) y el **mismo código** se ejecutará distribuido sobre decenas de nodos: el Driver coordinará, el Cluster Manager asignará recursos y los Executors procesarán las particiones en paralelo en máquinas distintas. Esa portabilidad es precisamente la gran ventaja de Spark frente a nuestro sistema actual.
>
> **Sobre la evaluación perezosa.** No es un retraso, es una **optimización**. Hoy hemos definido cuatro transformaciones sobre el fichero de entregas (`filter`, `map`, `map`, `reduceByKey`) y Spark no ha procesado ni un solo registro hasta que hemos pedido un resultado con `count()` o `collect()`. En la Spark UI se ve claramente: **un único job** ejecuta toda la cadena de operaciones de forma encadenada, sin materializar resultados intermedios y aplicando un `combine` local **antes** del shuffle. Si Spark ejecutara cada línea al instante, leería el fichero múltiples veces y enviaría datos innecesarios por la red. La evaluación perezosa es lo que le permite construir el **DAG**, optimizarlo y ejecutar el mínimo trabajo imprescindible.
>
> Quedo a su disposición para una demo en cuanto lo desee.
>
> Un saludo,  
> Equipo de Datos — LogiTrack S.A.

---
## 🛑 Cierre de la sesión

> Ejecuta esta celda **solo al terminar todos los ejercicios** para liberar los recursos de Spark.

In [19]:
spark.stop()
println("Sesión Spark detenida.")

Sesión Spark detenida.
